# Turkish BGE Reranker Experiment for Turkish Legal RAG

This notebook evaluates a Turkish fine-tuned Cross-Encoder reranker for the Turkish Legal RAG system.

The model used in this experiment is:

`seroe/bge-reranker-v2-m3-turkish-triplet`

This notebook keeps the same:
- retrieval corpus
- FAISS index
- BM25 setup
- test samples
- strict prompt
- LLM
- generation settings
- manual evaluation strategy

The only changed component is the reranker model.

In [1]:
!pip install -q -U sentence-transformers faiss-cpu rank-bm25 transformers accelerate bitsandbytes rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"
processed_path = f"{project_path}/data/processed"
faiss_path = f"{project_path}/outputs/faiss"
metrics_path = f"{project_path}/outputs/metrics"

print("Project path:", project_path)
print("Processed path:", processed_path)
print("FAISS path:", faiss_path)
print("Metrics path:", metrics_path)

Project path: /content/drive/MyDrive/turkish_legal_rag
Processed path: /content/drive/MyDrive/turkish_legal_rag/data/processed
FAISS path: /content/drive/MyDrive/turkish_legal_rag/outputs/faiss
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [4]:
import os
import re
import json
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

from sentence_transformers import CrossEncoder

In [5]:
chunks_df = pd.read_csv(f"{processed_path}/retrieval_corpus.csv")
test_qa_df = pd.read_csv(f"{processed_path}/test_qa.csv")

print("Chunks shape:", chunks_df.shape)
print("Test QA shape:", test_qa_df.shape)

chunks_df.head()

Chunks shape: (3775, 5)
Test QA shape: (1500, 2)


,chunk_id,source_context_id,source,chunk_text,chunk_len
0,chunk_000000,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,263
1,chunk_000001,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Dünya milletleri ailesinin eşit haklara sahip ...,194
2,chunk_000002,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Millet iradesinin mutlak üstünlüğü, egemenliği...",276
3,chunk_000003,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Kuvvetler ayrımının, Devlet organları arasında...",256
4,chunk_000004,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Hiçbir faaliyetin Türk milli menfaatlerinin, T...",370


In [6]:
test_qa_df.head()

,question,answer
0,Anayasanın 90. Maddesi Nasıl Uygulanır?,Milletlerarası antlaşmaların TBMM tarafından o...
1,Hukukta 'legitimate expectation' nedir?,"Legitimate expectation, bir kişinin belirli bi..."
2,"Anayasa madde 172'ye göre, devletin sanayi ve ...","Anayasa madde 172'ye göre, devlet, sanayi ve t..."
3,"Anayasa madde 158, uyuşmazlık mahkemesi'nin ku...","Anayasa madde 158'e göre, uyuşmazlık mahkemesi..."
4,"Bir grup avukat, Türkiye Büyük Millet Meclisi ...","Anayasanın 94. Maddesi, Türkiye Büyük Millet M..."


In [7]:
test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [8]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. LLM loading may be very slow or may fail.")

CUDA available: True
GPU: Tesla T4


In [9]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [11]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

index = faiss.read_index(f"{faiss_path}/baseline_faiss.index")

print("Embedding model loaded:", embedding_model_name)
print("Chunks:", chunks_df.shape)
print("FAISS vectors:", index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Chunks: (3775, 5)
FAISS vectors: 3775


In [12]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]

In [13]:
tokenized_corpus = [
    simple_turkish_tokenize(text)
    for text in chunks_df["chunk_text"].astype(str).tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready.")

BM25 ready.


In [14]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_retrieve_top_k(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(chunks_df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(chunks_df))
    ])

    tokenized_query = simple_turkish_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(tokenized_query))

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source": chunks_df.iloc[idx]["source"],
            "score": float(final_scores[idx]),
            "dense_score": float(dense_norm[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "chunk_text": chunks_df.iloc[idx]["chunk_text"]
        })

    return results

In [15]:
def detect_source_filter(query):
    q = str(query).lower()

    if "anayasa" in q or "anayasanın" in q:
        return "Türkiye Cumhuriyeti Anayasası"

    return None


def hybrid_retrieve_top_k_filtered(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    source_filter = detect_source_filter(query)

    candidate_df = chunks_df.copy()

    if source_filter is not None:
        candidate_df = candidate_df[
            candidate_df["source"].astype(str).str.lower() == source_filter.lower()
        ].reset_index(drop=True)

    if len(candidate_df) == 0:
        candidate_df = chunks_df.copy()

    candidate_texts = candidate_df["chunk_text"].astype(str).tolist()

    candidate_embeddings = embedding_model.encode(
        candidate_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(candidate_embeddings)

    temp_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    temp_index.add(candidate_embeddings)

    tokenized_candidate_corpus = [
        simple_turkish_tokenize(text)
        for text in candidate_texts
    ]

    temp_bm25 = BM25Okapi(tokenized_candidate_corpus)

    return hybrid_retrieve_top_k(
        query,
        model,
        temp_index,
        candidate_df,
        temp_bm25,
        k=k,
        alpha=alpha
    )

In [16]:
test_results = hybrid_retrieve_top_k_filtered(
    "Egemenlik kime aittir?",
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=5,
    alpha=0.5
)

for r in test_results:
    print("=" * 80)
    print(r["rank"], r["chunk_id"], r["score"], r["source"])
    print(r["chunk_text"][:500])

1 chunk_000269 0.9583597183227539 Türkiye Cumhuriyeti Anayasası
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
2 chunk_003519 0.8980596661567688 Türk Ceza Kanunu
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
3 chunk_002321 0.8044273853302002 Türk Medeni Kanunu
Madde 960- Ortaklık genel kurulunda rehinli pay senetlerini temsil etmek yetkisi, rehin 
alacaklısına değil, pay sahibine aittir.
II I. Yönetim ve ödeme
4 chunk_001360 0.744124710559845 Ceza Muhakemesi Kanunu
Madde 12 – (1) Davaya bakmak yetkisi, suçun işlendiği yer mahkemesine aittir.
5 chunk_000268 0.7346979975700378 Türkiye Cumhuriyeti Anayasası
Madde 5 – Devletin temel amaç ve görevleri, Türk milletinin bağımsızlığını ve bütünlüğünü, ülkenin bölünmezliğini, Cumhuriyeti ve demokrasiyi korumak, kişilerin ve toplumun refah, huz

## BGE Reranker

In [17]:
turkish_reranker_model_name = "seroe/bge-reranker-v2-m3-turkish-triplet"

# CPU kullanıyoruz ki Mistral GPU belleğiyle kavga etmesin.
# 20 test * 10 candidate için CPU yeterli olur.
turkish_reranker = CrossEncoder(
    turkish_reranker_model_name,
    device="cpu",
    max_length=512
)

print("Turkish BGE reranker loaded:", turkish_reranker_model_name)

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Turkish BGE reranker loaded: seroe/bge-reranker-v2-m3-turkish-triplet


In [18]:
def rerank_retrieved_chunks_turkish_bge(question, retrieved_results, top_k=None):
    pairs = [
        [question, item["chunk_text"]]
        for item in retrieved_results
    ]

    scores = turkish_reranker.predict(
        pairs,
        batch_size=4,
        show_progress_bar=False
    )

    scored_results = []

    for item, score in zip(retrieved_results, scores):
        scored_results.append({
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "original_rank": item["rank"],
            "original_hybrid_score": item["score"],
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "rerank_score": float(score),
            "chunk_text": item["chunk_text"]
        })

    scored_results = sorted(
        scored_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    final_results = []

    if top_k is None:
        top_k = len(scored_results)

    for new_rank, item in enumerate(scored_results[:top_k], start=1):
        item["rerank_rank"] = new_rank
        final_results.append(item)

    return final_results

In [20]:
def hybrid_retrieve_with_turkish_bge_fusion(
    query,
    model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5,
    hybrid_weight=0.7,
    rerank_weight=0.3
):
    candidates = hybrid_retrieve_top_k_filtered(
        query,
        model,
        index,
        chunks_df,
        bm25,
        k=candidate_k,
        alpha=alpha
    )

    reranked_all = rerank_retrieved_chunks_turkish_bge(
        question=query,
        retrieved_results=candidates,
        top_k=candidate_k
    )

    rerank_rank_map = {
        item["chunk_id"]: item["rerank_rank"]
        for item in reranked_all
    }

    rerank_score_map = {
        item["chunk_id"]: item["rerank_score"]
        for item in reranked_all
    }

    fused_results = []

    for item in candidates:
        chunk_id = item["chunk_id"]

        original_rank = item["rank"]
        rerank_rank = rerank_rank_map.get(chunk_id, candidate_k + 1)

        hybrid_rank_score = 1 / original_rank
        rerank_rank_score = 1 / rerank_rank

        fusion_score = (
            hybrid_weight * hybrid_rank_score
            + rerank_weight * rerank_rank_score
        )

        fused_results.append({
            "rank": None,
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "fusion_score": float(fusion_score),
            "original_rank": original_rank,
            "rerank_rank": rerank_rank,
            "rerank_score": float(rerank_score_map.get(chunk_id, 0.0)),
            "original_hybrid_score": item["score"],
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "chunk_text": item["chunk_text"]
        })

    fused_results = sorted(
        fused_results,
        key=lambda x: x["fusion_score"],
        reverse=True
    )

    final_results = []

    for new_rank, item in enumerate(fused_results[:final_k], start=1):
        item["rank"] = new_rank
        final_results.append(item)

    return final_results

In [21]:
CANDIDATE_K = 10
FINAL_CONTEXT_K = 3
ALPHA = 0.5

HYBRID_WEIGHT = 0.7
RERANK_WEIGHT = 0.3

print("Candidate K:", CANDIDATE_K)
print("Final context K:", FINAL_CONTEXT_K)
print("Alpha:", ALPHA)
print("Hybrid weight:", HYBRID_WEIGHT)
print("Rerank weight:", RERANK_WEIGHT)

Candidate K: 10
Final context K: 3
Alpha: 0.5
Hybrid weight: 0.7
Rerank weight: 0.3


In [22]:
sample_question = "Cumhurbaşkanının görev süresi kaç yıldır?"

turkish_bge_results = hybrid_retrieve_with_turkish_bge_fusion(
    sample_question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    candidate_k=CANDIDATE_K,
    final_k=FINAL_CONTEXT_K,
    alpha=ALPHA,
    hybrid_weight=HYBRID_WEIGHT,
    rerank_weight=RERANK_WEIGHT
)

print("QUESTION:")
print(sample_question)

for r in turkish_bge_results:
    print("=" * 80)
    print("New rank:", r["rank"])
    print("Chunk ID:", r["chunk_id"])
    print("Original rank:", r["original_rank"])
    print("Rerank rank:", r["rerank_rank"])
    print("Rerank score:", r["rerank_score"])
    print("Fusion score:", r["fusion_score"])
    print("Source:", r["source"])
    print(r["chunk_text"][:500])

QUESTION:
Cumhurbaşkanının görev süresi kaç yıldır?
New rank: 1
Chunk ID: chunk_000043
Original rank: 1
Rerank rank: 1
Rerank score: 0.9966253042221069
Fusion score: 1.0
Source: Türkiye Cumhuriyeti Anayasası
Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğrenim yapmış, milletvekili seçilme yeterliliğine sahip Türk vatandaşları arasından, doğrudan halk tarafından seçilir. Cumhurbaşkanının görev süresi beş yıldır. Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
New rank: 2
Chunk ID: chunk_000086
Original rank: 2
Rerank rank: 2
Rerank score: 0.9143514037132263
Fusion score: 0.5
Source: Türkiye Cumhuriyeti Anayasası
Seçimlerinin birlikte yenilenmesine karar verilen Meclisin ve Cumhurbaşkanının yetki ve görevleri, yeni Meclisin ve Cumhurbaşkanının göreve başlamasına kadar devam eder. Bu şekilde seçilen Meclis ve Cumhurbaşkanının görev süreleri de beş yıldır. İ. Milli Savunma 1. Başkomutanlık ve Genelkurmay Başkanlığı
New rank: 3
Chunk ID: chunk_000588
Original rank: 3
Rerank rank: 6
Re

In [23]:
pure_debug_candidates = hybrid_retrieve_top_k_filtered(
    sample_question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=CANDIDATE_K,
    alpha=ALPHA
)

pure_bge_reranked = rerank_retrieved_chunks_turkish_bge(
    question=sample_question,
    retrieved_results=pure_debug_candidates,
    top_k=CANDIDATE_K
)

for r in pure_bge_reranked:
    print("=" * 80)
    print("Rerank rank:", r["rerank_rank"])
    print("Original rank:", r["original_rank"])
    print("Chunk ID:", r["chunk_id"])
    print("Rerank score:", r["rerank_score"])
    print(r["chunk_text"][:400])

Rerank rank: 1
Original rank: 1
Chunk ID: chunk_000043
Rerank score: 0.9966253042221069
Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğrenim yapmış, milletvekili seçilme yeterliliğine sahip Türk vatandaşları arasından, doğrudan halk tarafından seçilir. Cumhurbaşkanının görev süresi beş yıldır. Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
Rerank rank: 2
Original rank: 2
Chunk ID: chunk_000086
Rerank score: 0.9143514037132263
Seçimlerinin birlikte yenilenmesine karar verilen Meclisin ve Cumhurbaşkanının yetki ve görevleri, yeni Meclisin ve Cumhurbaşkanının göreve başlamasına kadar devam eder. Bu şekilde seçilen Meclis ve Cumhurbaşkanının görev süreleri de beş yıldır. İ. Milli Savunma 1. Başkomutanlık ve Genelkurmay Başkanlığı
Rerank rank: 3
Original rank: 4
Chunk ID: chunk_000025
Rerank score: 0.2545464336872101
Türkiye Büyük Millet Meclisi Başkanlık Divanı için, bir yasama döneminde iki seçim yapılır. (Değişik ikinci cümle: 7/5/2010-5982/10 md.) İlk seçilenlerin görev süresi ik

In [24]:
base_eval_questions = [
    {
        "question": "Egemenlik kime aittir?",
        "expected_answer": "Egemenlik kayıtsız şartsız Milletindir."
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "expected_answer": "Türkiye Devleti bir Cumhuriyettir."
    },
    {
        "question": "Cumhurbaşkanının görev süresi kaç yıldır?",
        "expected_answer": "Cumhurbaşkanının görev süresi beş yıldır."
    },
    {
        "question": "Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?",
        "expected_answer": "Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir."
    }
]

base_eval_df = pd.DataFrame(base_eval_questions)
base_eval_df

,question,expected_answer
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...


In [26]:
def build_strict_rag_prompt(question, retrieved_contexts):
    context_text = "\n\n".join([
        f"[Context {i+1}]\n{ctx}"
        for i, ctx in enumerate(retrieved_contexts)
    ])

    prompt = f"""
Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.

Kurallar:
- Cevabı SADECE verilen bağlama göre ver.
- Bağlamda açıkça yazmayan çıkarımları yapma.
- Birden fazla bağlam çelişirse en doğrudan cevap veren bağlamı kullan.
- Cevap Türkçe olmalı.
- Cevap kısa ve net olmalı.
- İngilizce açıklama, "Therefore", "Context 1" gibi ifadeler yazma.

Bağlam:
{context_text}

Soru:
{question}

Kısa cevap:
"""
    return prompt.strip()


def clean_generated_answer(text):
    text = str(text)

    if "Kısa cevap:" in text:
        text = text.split("Kısa cevap:")[-1].strip()

    if "Detaylı cevap:" in text:
        text = text.split("Detaylı cevap:")[0].strip()

    return text.strip()

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

llm_model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("LLM loaded:", llm_model_name)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LLM loaded: mistralai/Mistral-7B-Instruct-v0.2


In [32]:
def generate_answer(prompt, max_new_tokens=180):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in generated_text:
        generated_text = generated_text.split("Answer:")[-1].strip()

    return generated_text

In [33]:
base_eval_questions = [
    {
        "question": "Egemenlik kime aittir?",
        "expected_answer": "Egemenlik kayıtsız şartsız Milletindir."
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "expected_answer": "Türkiye Devleti bir Cumhuriyettir."
    },
    {
        "question": "Cumhurbaşkanının görev süresi kaç yıldır?",
        "expected_answer": "Cumhurbaşkanının görev süresi beş yıldır."
    },
    {
        "question": "Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?",
        "expected_answer": "Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir."
    }
]

base_eval_df = pd.DataFrame(base_eval_questions)
base_eval_df

,question,expected_answer
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...


In [34]:
turkish_bge_small_results = []

for _, row in base_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["expected_answer"]

    retrieved = hybrid_retrieve_with_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_strict_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)

    turkish_bge_small_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_context": retrieved[0]["chunk_text"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_rerank_score": retrieved[0]["rerank_score"],
        "top1_fusion_score": retrieved[0]["fusion_score"]
    })

turkish_bge_small_results_df = pd.DataFrame(turkish_bge_small_results)
turkish_bge_small_results_df

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Sen Türk hukuk metinleri için çalışan dikkatli...,Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin...",1,1,0.984422,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Sen Türk hukuk metinleri için çalışan dikkatli...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,1,3,0.008710,0.8
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Sen Türk hukuk metinleri için çalışan dikkatli...,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,1,0.996625,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Sen Türk hukuk metinleri için çalışan dikkatli...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,1,0.995044,1.0


In [35]:
for i, row in turkish_bge_small_results_df.iterrows():
    print("=" * 100)
    print("INDEX:", i)

    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"])

    print("\nTOP1 CHUNK ID:", row["top1_chunk_id"])
    print("TOP1 ORIGINAL RANK:", row["top1_original_rank"])
    print("TOP1 RERANK RANK:", row["top1_rerank_rank"])
    print("TOP1 RERANK SCORE:", row["top1_rerank_score"])
    print("TOP1 FUSION SCORE:", row["top1_fusion_score"])

INDEX: 0

QUESTION:
Egemenlik kime aittir?

EXPECTED:
Egemenlik kayıtsız şartsız Milletindir.

CLEAN GENERATED:
Egemenlik Türk Milleti'ne aittir.

TOP1 CHUNK ID: chunk_000269
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 1
TOP1 RERANK SCORE: 0.9844223260879517
TOP1 FUSION SCORE: 1.0
INDEX: 1

QUESTION:
Türkiye Cumhuriyetinin yönetim şekli nedir?

EXPECTED:
Türkiye Devleti bir Cumhuriyettir.

CLEAN GENERATED:
Türkiye Cumhuriyetinin yönetim şekli bölünmez bir bütün olduğu Türk Devletidir.

TOP1 CHUNK ID: chunk_000000
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 3
TOP1 RERANK SCORE: 0.008710420690476894
TOP1 FUSION SCORE: 0.7999999999999999
INDEX: 2

QUESTION:
Cumhurbaşkanının görev süresi kaç yıldır?

EXPECTED:
Cumhurbaşkanının görev süresi beş yıldır.

CLEAN GENERATED:
Cumhurbaşkanının görev süresi beş yıldır.

TOP1 CHUNK ID: chunk_000043
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 1
TOP1 RERANK SCORE: 0.9966253042221069
TOP1 FUSION SCORE: 1.0
INDEX: 3

QUESTION:
Bir kimse en fazla kaç defa Cumhurbaşk

In [36]:
manual_scores_turkish_bge_small = [
    1.0,
    0.0,
    1.0,
    1.0
]

turkish_bge_small_results_df["manual_score"] = manual_scores_turkish_bge_small

turkish_bge_small_score = turkish_bge_small_results_df["manual_score"].mean()

print("Turkish BGE Reranker 4-question score:", turkish_bge_small_score)
turkish_bge_small_results_df

Turkish BGE Reranker 4-question score: 0.75


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,manual_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Sen Türk hukuk metinleri için çalışan dikkatli...,Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin...",1,1,0.984422,1.0,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Sen Türk hukuk metinleri için çalışan dikkatli...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,1,3,0.008710,0.8,0.0
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Sen Türk hukuk metinleri için çalışan dikkatli...,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,1,0.996625,1.0,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Sen Türk hukuk metinleri için çalışan dikkatli...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,1,0.995044,1.0,1.0


In [37]:
turkish_bge_small_results_df.to_csv(
    f"{metrics_path}/turkish_bge_reranker_4question_results.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Turkish BGE Reranker Fusion RAG - 4 Question Sanity Evaluation",
    "manual_accuracy": turkish_bge_small_score,
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT
}]).to_csv(
    f"{metrics_path}/turkish_bge_reranker_4question_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Turkish BGE 4-question results saved.")

Turkish BGE 4-question results saved.


In [38]:
test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [39]:
turkish_bge_test_results = []

for _, row in tqdm(test_eval_df.iterrows(), total=len(test_eval_df)):
    question = row["question"]
    expected_answer = row["answer"]

    retrieved = hybrid_retrieve_with_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_strict_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)

    turkish_bge_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_context": retrieved[0]["chunk_text"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_rerank_score": retrieved[0]["rerank_score"],
        "top1_fusion_score": retrieved[0]["fusion_score"],
        "retrieved_contexts": "\n\n".join(contexts)
    })

turkish_bge_test_results_df = pd.DataFrame(turkish_bge_test_results)
turkish_bge_test_results_df.head()

100%|██████████| 20/20 [07:29<00:00, 22.46s/it]


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000210,Türkiye Cumhuriyeti Anayasası,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,1,4,0.007053,0.7750,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 10. Maddesi, herkesin eşittirliği s...",chunk_000188,Türkiye Cumhuriyeti Anayasası,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",1,4,0.000305,0.7750,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayatın sınırlanması, Anayasanın 17. Maddesine...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1,1,0.385170,1.0000,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Sen Türk hukuk metinleri için çalışan dikkatli...,Geçici madde 20 1987 yılında eklendi.,chunk_000600,Bilgi Edinme Kanunu,Madde 20- Açıklanması veya zamanından önce açı...,1,8,0.000864,0.7375,Madde 20- Açıklanması veya zamanından önce açı...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayır, TCK 121 ihlali sabit değildir. (Madde 1...",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,1,6,0.000157,0.7500,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...


In [40]:
turkish_bge_test_results_df.to_csv(
    f"{metrics_path}/turkish_bge_reranker_fusion_rag_testset_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Turkish BGE reranker fusion test generation results saved.")

Turkish BGE reranker fusion test generation results saved.


In [41]:
for i, row in turkish_bge_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)

    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"][:1000])

    print("\nTOP1 CHUNK ID:", row["top1_chunk_id"])
    print("TOP1 SOURCE:", row["top1_source"])
    print("TOP1 ORIGINAL RANK:", row["top1_original_rank"])
    print("TOP1 RERANK RANK:", row["top1_rerank_rank"])
    print("TOP1 RERANK SCORE:", row["top1_rerank_score"])
    print("TOP1 FUSION SCORE:", row["top1_fusion_score"])

INDEX: 0

QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

CLEAN GENERATED:
Anayasanın 101. Maddesiyle ilgili tartışmalar, seçimlerin geriye bırakılması ve ikinci oylama hakkındadır.

TOP1 CHUNK ID: chunk_000210
TOP1 SOURCE: Türkiye Cumhuriyeti Anayasası
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 4
TOP1 RERANK SCORE: 0.007053057197481394
TOP1 FUSION SCORE: 0.7749999999999999
INDEX: 1

QUESTION:
Bir grup vatandaş, belirli bir etnik grubun diğerlerinden daha fazla hakka sahip olması için imza kampanyası başlatmıştır. Bu durum Anayasanın 10. Maddesi ile nasıl çelişir?

EXPECTED:
Anayasanın 10. Maddesi, herkesin kanun önünde eşit olduğunu belirtir. Bu tür bir imza kampanyası Anayasa'ya aykırıdır.

CLEAN GENERATED:
Anayasanın 10. Maddesi, herkesin eşittirliği sağlayan hükümüdür. Etnik gruplar arasında herkesin eşit haklarının sağlanması istenirse, bu, yeni yazılan ve mevcut 

In [42]:
turkish_bge_test_results_df["is_valid_sample"] = True

# 03 ve 04 ile aynı evaluation setup:
# index 19 invalid sample olarak skora dahil edilmiyor.
turkish_bge_test_results_df.loc[19, "is_valid_sample"] = False

manual_scores_turkish_bge = [
    0.0,
    0.5,
    0.5,
    0.0,
    0.5,
    0.0,
    0.0,
    1.0,
    0.0,
    0.0,
    0.5,
    0.0,
    0.0,
    1.0,
    0.5,
    0.5,
    0.5,
    0.0,
    0.5,
    0.0
]

turkish_bge_test_results_df["manual_score"] = manual_scores_turkish_bge

valid_turkish_bge_df = turkish_bge_test_results_df[
    turkish_bge_test_results_df["is_valid_sample"] == True
]

turkish_bge_test_score = valid_turkish_bge_df["manual_score"].mean()

print("Turkish BGE Reranker Fusion RAG Test Score:", turkish_bge_test_score)
print("Valid sample count:", len(valid_turkish_bge_df))
print("Total sample count:", len(turkish_bge_test_results_df))

Turkish BGE Reranker Fusion RAG Test Score: 0.3157894736842105
Valid sample count: 19
Total sample count: 20


In [43]:
turkish_bge_test_results_df.to_csv(
    f"{metrics_path}/turkish_bge_reranker_fusion_rag_testset_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Turkish BGE Reranker Fusion RAG - Hybrid Retrieval + seroe/bge-reranker-v2-m3-turkish-triplet + Rank Fusion + Strict Prompt",
    "manual_accuracy": turkish_bge_test_score,
    "valid_sample_count": len(valid_turkish_bge_df),
    "total_sample_count": len(turkish_bge_test_results_df),
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT
}]).to_csv(
    f"{metrics_path}/turkish_bge_reranker_fusion_rag_testset_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Turkish BGE reranker fusion scored results saved.")

Turkish BGE reranker fusion scored results saved.


In [44]:
test_comparison_df = pd.DataFrame([
    {
        "method": "Base RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Base prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "Strict Prompt RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "FlashRank Fusion Reranker RAG",
        "retrieval_setup": "Hybrid top-10 + FlashRank reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.184211
    },
    {
        "method": "Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": turkish_bge_test_score
    }
])

test_comparison_df

,method,retrieval_setup,prompt,manual_accuracy
0,Base RAG,"Hybrid retrieval top-5, context top-3",Base prompt,0.263158
1,Strict Prompt RAG,"Hybrid retrieval top-5, context top-3",Strict prompt,0.263158
2,FlashRank Fusion Reranker RAG,Hybrid top-10 + FlashRank reranker + rank fusi...,Strict prompt,0.184211
3,Turkish BGE Reranker Fusion RAG,Hybrid top-10 + Turkish BGE reranker + rank fu...,Strict prompt,0.315789


In [45]:
test_comparison_df.to_csv(
    f"{metrics_path}/rag_reranker_model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Reranker model comparison saved.")

Reranker model comparison saved.
